<a href="https://colab.research.google.com/github/IsabellaCada2506/Inteligencia_Artificial/blob/main/quices/03_lab_warehouse_mdp_estudiantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [19]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: Ajusta estas coordenadas exactas según tu imagen
        self.start = (4, 0)
        self.walls = {(1, 2), (2, 2), (3, 2)} # estanterías / paredes
        self.slippery_states = {(0, 4), (1, 4), (2, 4)} # celdas de piso resbaloso

        self.terminal_states = {
            (0, 5): 10.0,  # zona de entrega +10
            (4, 5): 2.0,   # estación de carga +2
            (0, 3): -10.0  # peligro mortal -10
        }

        self.danger_states = {
            (1, 3): -3.0,  # peligros no terminales -3
            (2, 3): -3.0
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # Probabilidades configurables para facilitar los experimentos
        self.normal_probs = [0.90, 0.05, 0.05]
        self.slippery_probs = [0.60, 0.20, 0.20]

        self.actions = [
            (-1, 0),  # UP
            (1, 0),   # DOWN
            (0, -1),  # LEFT
            (0, 1),   # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        return (
            0 <= r < self.height
            and 0 <= c < self.width
            and state not in self.walls
        )

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            probs = self.slippery_probs
        else:
            probs = self.normal_probs

        dr, dc = action
        intended = action
        left = (-dc, dr)
        right = (dc, -dr)

        directions = [intended, left, right]
        transitions = {}

        for direction, probability in zip(directions, probs):
            nr = state[0] + direction[0]
            nc = state[1] + direction[1]
            next_state = (nr, nc)

            if not self.is_valid_state(next_state):
                next_state = state # Si choca, se queda en el mismo lugar

            transitions[next_state] = (
                transitions.get(next_state, 0) + probability
            )

        return list(transitions.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [20]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 27
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [21]:
def expected_next_value(grid, state, action, V):
    total = 0.0
    for next_state, probability in grid.get_transition_probs(state, action):
        total += probability * V[next_state]
    return total

def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    states = grid.states()
    V = {s: 0.0 for s in states}

    for iteration in range(1, max_iter + 1):
        new_V = {}
        delta = 0.0

        for state in states:
            if grid.is_terminal(state):
                new_V[state] = grid.get_reward(state)
                continue

            action_values = []
            for action in grid.actions:
                value = expected_next_value(grid, state, action, V)
                action_values.append(value)

            best_value = max(action_values)
            new_V[state] = grid.get_reward(state) + grid.gamma * best_value

            delta = max(delta, abs(new_V[state] - V[state]))

        V = new_V
        if delta < threshold:
            return V, iteration

    return V, max_iter

def extract_policy(grid, V):
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        best_action = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
        policy[state] = best_action
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [22]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    states = grid.states()
    V = {s: 0.0 for s in states}

    for iteration in range(max_iter):
        delta = 0.0
        new_V = V.copy()

        for state in states:
            if grid.is_terminal(state):
                new_V[state] = grid.get_reward(state)
                continue

            action = policy[state]
            expected_value = expected_next_value(grid, state, action, V)
            new_V[state] = grid.get_reward(state) + grid.gamma * expected_value

            delta = max(delta, abs(new_V[state] - V[state]))

        V = new_V
        if delta < threshold:
            break
    return V

def policy_improvement(grid, V):
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        best_action = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
        policy[state] = best_action
    return policy

def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {}
    for state in grid.states():
        if not grid.is_terminal(state):
            policy[state] = grid.actions[0]

    history = []
    for iteration in range(max_iter):
        V = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V)
        history.append(iteration + 1)

        if new_policy == policy:
            return new_policy, V, history

        policy = new_policy

    return policy, V, history



## Parte 4 — Visualización y comparación


In [23]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))

def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f} ")
            else:
                row.append(f" {ARROWS.get(policy.get(s, None), '?')} ")
        print(" | ".join(row))

# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)

# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")

=== VALUE ITERATION ===
Iteraciones: 28

Valores:
 -5.630 |  -5.274 |  -5.793 | -10.000 |  +6.441 | +10.000
 -5.111 |  -4.642 |   WALL   |  +0.505 |  +4.897 |  +7.665
 -4.533 |  -3.954 |   WALL   |  -0.249 |  +3.323 |  +5.611
 -3.890 |  -3.176 |   WALL   |  +0.830 |  +2.285 |  +3.820
 -3.176 |  -2.294 |  -1.293 |  -0.218 |  +0.931 |  +2.000

Política:
 ↓  |  ↓  |  ←  | -10  |  →  | +10 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 →  |  →  |  →  |  →  |  ↑  | +2 

=== POLICY ITERATION ===
Historia: [1, 2, 3, 4, 5, 6]

Valores:
 -5.630 |  -5.274 |  -5.793 | -10.000 |  +6.441 | +10.000
 -5.111 |  -4.642 |   WALL   |  +0.505 |  +4.897 |  +7.665
 -4.533 |  -3.954 |   WALL   |  -0.249 |  +3.323 |  +5.611
 -3.890 |  -3.176 |   WALL   |  +0.830 |  +2.285 |  +3.820
 -3.176 |  -2.294 |  -1.293 |  -0.218 |  +0.931 |  +2.000

Política:
 ↓  |  ↓  |  ←  | -10  |  →  | +10 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


#SOLUCIÓN:

1. Desde START, ¿el robot busca la entrega +10 o prefiere la estación de carga +2?
Depende fuertemente de la distancia a ambas terminales y del living_reward. Con un costo de -1 por paso, a menudo el robot prefiere la estación de carga si está mucho más cerca, ya que ir hacia el +10 restaría tantos puntos por el camino (y por el factor de descuento $\gamma$) que la recompensa final neta sería menor a +2.

2. ¿Por qué una recompensa menor podría ser óptima?
Porque en los MDP optimizamos la recompensa acumulada y descontada. Una meta lejana de +10 penalizada por muchos pasos de -1 (y rebajada por $\gamma^n$) resulta matemáticamente menos valiosa que un +2 seguro y cercano a solo un par de pasos.

3. ¿En qué estados el piso resbaloso cambia la decisión?
Principalmente cerca de las celdas de peligro (-3 y -10). Como el piso resbaloso tiene un 40% de probabilidad de desviación (20% a cada lado), el robot elegirá políticas conservadoras, evitando caminar directamente al borde de los peligros y prefiriendo alejarse o chocar intencionalmente contra las paredes (para resetear el deslizamiento) en lugar de arriesgarse a caer en el peligro mortal.

4. ¿Qué papel cumple el costo por paso -1?
Actúa como un incentivo para la eficiencia. Obliga al robot a buscar el camino más corto hacia una meta positiva. Sin este costo (si fuera 0), el robot podría deambular indefinidamente siempre y cuando eventualmente llegue a una recompensa.

5. ¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?
Porque la dinámica del mundo cambia según la ubicación física del agente. Las zonas resbalosas alteran la probabilidad de éxito de una acción (0.60 en lugar de 0.90), y la cercanía a obstáculos o bordes significa que los intentos de movimiento pueden rebotar y mantener al agente en el mismo estado $s$.


##Experimento A

In [24]:
grid_A = WarehouseMDP()
grid_A.living_reward = -0.1  # Menos penalización por moverse

V_A, n_A = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)

print("=== Experimento A: living_reward = -0.1 ===")
print("Iteraciones:", n_A)
print("\nPolítica con living_reward = -0.1:")
print_policy(grid_A, pi_A)

=== Experimento A: living_reward = -0.1 ===
Iteraciones: 28

Política con living_reward = -0.1:
 ↓  |  ↓  |  ←  | -10  |  →  | +10 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 →  |  →  |  →  |  →  |  ↑  | +2 


Al reducir la penalización por paso, el robot estará dispuesto a caminar más lejos para alcanzar la entrega de +10, abandonando la meta menor de +2.

##Experimento B (Piso muy resbaloso)

In [25]:
grid_B = WarehouseMDP()
# Repartimos el sobrante equitativamente: 0.40 éxito, 0.30 izq, 0.30 der
grid_B.slippery_probs = [0.40, 0.30, 0.30]

V_B, n_B = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)

print("=== Experimento B: Piso muy resbaloso (0.40 éxito) ===")
print_policy(grid_B, pi_B)

=== Experimento B: Piso muy resbaloso (0.40 éxito) ===
 ↓  |  ↓  |  ←  | -10  |  →  | +10 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 →  |  →  |  →  |  →  |  ↑  | +2 


##Experimento C (Más paciencia)

In [26]:
grid_C = WarehouseMDP()
grid_C.gamma = 0.99  # Más visión a futuro

V_C, n_C = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)

print("=== Experimento C: Gamma = 0.99 ===")
print_policy(grid_C, pi_C)

=== Experimento C: Gamma = 0.99 ===
 ↓  |  ↓  |  ←  | -10  |  →  | +10 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 ↓  |  ↓  |  #  |  →  |  →  |  ↑ 
 →  |  →  |  →  |  →  |  ↑  | +2 


Sí, al aumentar $\gamma$, la política valora mucho más la recompensa +10 distante porque el castigo por retraso (el descuento temporal) es casi inexistente.

##Bonus (Umbral de Cambio de Decisión)

In [27]:
def find_threshold():
    rewards = np.linspace(-1.5, 0.0, 150)
    policies = []

    grid_test = WarehouseMDP()
    start_state = grid_test.start

    for r in rewards:
        grid_test.living_reward = r
        V_test, _ = value_iteration(grid_test)
        pi_test = extract_policy(grid_test, V_test)
        # Registramos qué acción toma desde START
        policies.append(pi_test[start_state])

    # Encontrar el punto de quiebre donde la acción cambia
    for i in range(1, len(rewards)):
        if policies[i] != policies[i-1]:
            print(f"El cambio de política ocurre aproximadamente en living_reward = {rewards[i]:.3f}")
            print(f"Acción antes del umbral: {ARROWS.get(policies[i-1])}")
            print(f"Acción después del umbral: {ARROWS.get(policies[i])}")
            return

find_threshold()